# Evaluate Performance Using CatBoost

In [1]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [3]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [4]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [5]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
2014,165.161,1310.0,11.0,204.30000,1012.728952,2907.0,5351.0,14948.5,12732.0,9699.0,...,0.000002,0.132912,5.350128,1825.0,1533.0,79989423.5,4.725000e+03,3.675000e+03,-316.669417,0.925926
1334,121.954,377.0,93.0,173.10000,1363.117917,2946.0,4692.0,15514.0,NaN,NaN,...,0.000251,0.186936,34.724747,1220.0,1159.0,72791688.0,1.819300e+04,1.502900e+04,-11.702699,0.869565
5898,81.145,179.6,20.0,133.30000,115.056783,2186.0,4889.0,14703.5,11544.5,12228.0,...,0.000093,0.254898,3.850726,1792.0,1728.0,71885411.5,1.683745e+06,1.515370e+06,-19.904327,0.933333
2314,245.918,758.0,183.0,168.40001,1296.518526,1623.0,3284.5,14641.0,13889.5,9824.0,...,0.000001,0.145714,0.000000,1782.0,1188.0,48088364.5,4.117773e+04,2.470664e+04,-35.600384,0.900000
5612,113.106,553.0,49.0,174.00000,952.729122,3244.0,5786.0,15102.0,13187.5,9932.0,...,0.000538,0.301439,10.106505,1495.0,1430.0,87380172.0,1.263721e+06,1.029699e+06,-5.551639,0.851852


In [6]:
train_df.shape

(6523, 33)

In [7]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

### Preprocess Data - Catboost doesn't require Imputation or Scaling, can essentially skip this step.

## **Feature Selection** 

In [8]:
# Will need to do different feature selection again for CatBoost since preprocessing steps are different than SVR.
# Conduct feature selection using shap_select again.

def perform_feature_selection(X_train, y_train):
    results_dict = {}

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)
    X_te_df = pd.DataFrame(X_te, columns=X_train.columns, index=y_te.index)
    
    
    for target in y_train.columns:
        model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
        model.fit(X_tr, y_tr[target], eval_set=[(X_te, y_te[target])])

        selected_df = shap_select(model, X_te_df, y_te[target], task="regression", threshold=0.05)
        results_dict[target] = selected_df

    return results_dict  

In [15]:
results = perform_feature_selection(X_train, y_train)

[0]	validation_0-rmse:58.85491
[1]	validation_0-rmse:49.90454
[2]	validation_0-rmse:43.98121
[3]	validation_0-rmse:39.98657
[4]	validation_0-rmse:37.09691
[5]	validation_0-rmse:35.05922
[6]	validation_0-rmse:34.03959
[7]	validation_0-rmse:33.39492
[8]	validation_0-rmse:32.66944
[9]	validation_0-rmse:31.96229
[10]	validation_0-rmse:31.51794
[11]	validation_0-rmse:31.27404
[12]	validation_0-rmse:31.06127
[13]	validation_0-rmse:30.93100
[14]	validation_0-rmse:30.66770
[15]	validation_0-rmse:30.61829
[16]	validation_0-rmse:30.24094
[17]	validation_0-rmse:30.12736
[18]	validation_0-rmse:29.86075
[19]	validation_0-rmse:29.73168
[20]	validation_0-rmse:29.58639
[21]	validation_0-rmse:29.61858
[22]	validation_0-rmse:29.61217
[23]	validation_0-rmse:29.52397
[24]	validation_0-rmse:29.41544
[25]	validation_0-rmse:29.36100
[26]	validation_0-rmse:29.31721
[27]	validation_0-rmse:29.21089
[28]	validation_0-rmse:29.19532
[29]	validation_0-rmse:29.12903
[30]	validation_0-rmse:28.97629
[31]	validation_0-

In [17]:
results

{'Total Alkalinity':                         feature name    t-value  stat.significance  \
 0                 cec_pH_interaction  15.673946       7.361350e-51   
 1                   skin_temperature  15.164197       5.871305e-48   
 2                  flow_accumulation  12.205200       1.616905e-32   
 3                          elevation   7.912707       5.352284e-15   
 4                                nir   7.869746       7.430035e-15   
 5                             swir16   6.080586       1.571873e-09   
 6            total_precipitation_sum   5.280180       1.511775e-07   
 7               NDVI_LST_interaction   4.425181       1.044363e-05   
 8                              MNDWI   3.663210       2.591170e-04   
 9              total_evaporation_sum   3.185135       1.481363e-03   
 10          Land Surface Temperature   2.843609       4.530778e-03   
 11                    temperature_2m   2.369782       1.794457e-02   
 12             volumetric_soil_water   1.667729       9.

In [9]:
total_alk_feats = ["cec_pH_interaction", "skin_temperature", "flow_accumulation", "elevation", "nir",
                   "swir16", "total_precipitation_sum", "NDVI_LST_interaction", "MNDWI", "total_evaporation_sum",
                   "Land Surface Temperature", "temperature_2m"]

len(total_alk_feats)

12

In [10]:
elec_cond_feats = [
    "skin_temperature",
    "NDVI",
    "elevation",
    "phosphorous",
    "total_evaporation_sum",
    "flow_acc_clay_interaction",
    "clay",
    "pH",
    "precipitation",
    "phosphorous_pH_interaction",
    "pet",
    "cec_clay_ratio",
    "cec_pH_interaction"
]

len(elec_cond_feats)

13

In [11]:
drp_feats = [
    "phosphorous_pH_interaction",
    "cec_pH_interaction",
    "pet",
    "flow_accumulation",
    "total_precipitation_sum",
    "elevation",
    "EVI",
    "nir",
    "flow_acc_phosphorous_interaction",
    "Land Surface Temperature",
    "cec_clay_ratio"
]

len(drp_feats)

11

In [12]:
selected_feats = set(list(total_alk_feats + elec_cond_feats + drp_feats))

len(selected_feats)

23